# Примеры компаний: путь от ФП к дефолту

Берём компании из таблицы дефолтов и смотрим, какие факторы проблемности (ФП) были зафиксированы в CRM до и после дефолта.

In [ ]:
import pandas as pd
import numpy as np

# Дефолты
df_def = pd.read_pickle("../sources/data_defaults.pkl")
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
for col in ["start_date", "cure_date", "finish_date"]:
    df_def[col] = pd.to_datetime(df_def[col], dayfirst=True, errors="coerce")
print(f"Дефолты: {len(df_def)} строк")

# CRM
df_crm = pd.read_csv("../sources/data_crm.csv", sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
print(f"CRM: {len(df_crm):,} строк")

## Функция для просмотра компании

In [ ]:
def show_company(inn):
    """Показывает информацию о дефолте и все уникальные ФП из CRM для компании."""

    # --- Дефолт ---
    d = df_def[df_def["inn"] == inn]
    if d.empty:
        print(f"ИНН {inn} не найден в таблице дефолтов.")
        return

    print("=" * 70)
    row = d.iloc[0]
    name = df_crm.loc[df_crm["X_INN"] == inn, "NAME_1"].dropna().iloc[0] if (df_crm["X_INN"] == inn).any() else "—"
    print(f"Компания: {name}")
    print(f"ИНН:      {inn}")
    print(f"Дефолт:   {row['start_date']:%d.%m.%Y}   Причина: {row['default_reason']}")
    if pd.notna(row.get("cure_date")):
        print(f"Cure:     {row['cure_date']:%d.%m.%Y}")
    if pd.notna(row.get("finish_date")):
        print(f"Finish:   {row['finish_date']:%d.%m.%Y}")
    print("=" * 70)

    # --- ФП из CRM ---
    crm = df_crm[df_crm["X_INN"] == inn].copy()
    if crm.empty:
        print("В CRM записей по этому ИНН нет.")
        return

    # Дедупликация по содержанию (ROW_ID у дублей разный, остальное совпадает)
    cols_no_id = [c for c in crm.columns if c != "ROW_ID"]
    crm_dedup = crm.drop_duplicates(subset=cols_no_id)

    show_cols = [
        "ROW_ID", "NUMBER_FP_SFP", "TYPE_FP", "VAL", "VAL_1",
        "IDENTIFICATION_DATE", "AGREEMENT_NUM", "SCRIPT",
    ]
    show_cols = [c for c in show_cols if c in crm_dedup.columns]
    result = crm_dedup[show_cols].sort_values("IDENTIFICATION_DATE").reset_index(drop=True)

    print(f"\nСтрок в CRM: {len(crm)},  после дедупликации: {len(crm_dedup)}")
    print(f"Уникальных ФП (NUMBER_FP_SFP): {crm_dedup['NUMBER_FP_SFP'].nunique()}")
    display(result)

## Выбор компаний для примеров

Берём несколько компаний из дефолтов, у которых есть записи в CRM.

In [ ]:
# ИНН дефолтных компаний, которые есть в CRM
def_inns = set(df_def["inn"].dropna())
crm_inns = set(df_crm["X_INN"].dropna())
common_inns = sorted(def_inns & crm_inns)

print(f"Дефолтных компаний:          {len(def_inns)}")
print(f"Из них есть в CRM:           {len(common_inns)}")
print(f"\nПервые 20 ИНН для выбора:")
for i, inn in enumerate(common_inns[:20], 1):
    n_fp = df_crm[df_crm['X_INN'] == inn]['NUMBER_FP_SFP'].nunique()
    print(f"  {i:2d}. {inn}  ({n_fp} уник. ФП)")

## Примеры

In [ ]:
# Подставь нужные ИНН
show_company(common_inns[0])

In [ ]:
show_company(common_inns[1])

In [ ]:
show_company(common_inns[2])